# Understanding ORB: Oriented FAST and Rotated BRIEF

*A complete guide to real-time feature detection and matching*

---

**ORB (Oriented FAST and Rotated BRIEF)** is a fast, robust, and patent-free alternative to SIFT and SURF. Designed for real-time applications, ORB combines the speed of FAST corner detection with the efficiency of binary BRIEF descriptors, adding rotation invariance to both.

This notebook provides a detailed walkthrough of the ORB algorithm with mathematical explanations and visual examples.

## Table of Contents

1. [Overview](#overview)
2. [Detection Phase](#detection-phase)
   - Step 1: Scale-Space Pyramid
   - Step 2: FAST Corner Detection
   - Step 3: Harris Corner Response
   - Step 4: Orientation Assignment
3. [Description Phase](#description-phase)
   - Step 5: rBRIEF Descriptor
   - Step 6: Hamming Distance Matching
4. [ORB vs SIFT Comparison](#orb-vs-sift-comparison)

## Overview

ORB operates in two main phases:

| Phase | Step | Description | Math |
|-------|------|-------------|------|
| Detection | 1 | Scale-Space Pyramid | `scale_n = 1/1.2^n` |
| Detection | 2 | FAST Corner Detection | 16-pixel Bresenham circle |
| Detection | 2.1 | High-Speed Test | Test 4 cardinal pixels |
| Detection | 2.2 | Full Contiguous Test | 9+ contiguous B or D |
| Detection | 3 | Harris Corner Response | `R = det(M) - k·trace(M)²` |
| Detection | 4 | Orientation Assignment | `θ = atan2(m₀₁, m₁₀)` |
| Description | 5 | rBRIEF Descriptor | 256-bit rotated binary pattern |
| Description | 6 | Hamming Distance Matching | XOR + popcount |

## Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy.ndimage import gaussian_filter, sobel
from PIL import Image
from IPython.display import display, Markdown
import os

# For better plots
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['figure.dpi'] = 100

print("Setup complete!")

## Load Input Image

In [ ]:
# Load the input image
img_path = 'images/input_image.jpg'
img = np.array(Image.open(img_path).convert('L')) / 255.0  # Normalize to 0-1
H, W = img.shape

plt.figure(figsize=(10, 8))
plt.imshow(img, cmap='gray')
plt.title(f'Input Image: {W} × {H} pixels', fontsize=14, fontweight='bold')
plt.axis('off')
plt.show()

print(f"Image shape: {img.shape}")

---
# 1️⃣ DETECTION PHASE

**Goal**: Find corner points that are stable across scale and rotation.

```
INPUT: Image (H × W)
        ↓
Step 1: Build Scale-Space Pyramid (8 levels, factor 1.2)
        ↓
Step 2: FAST Corner Detection (16-pixel circle test)
        ├── Step 2.1: High-Speed Test (4 pixels)
        └── Step 2.2: Full Contiguous Test (16 pixels)
        ↓
Step 3: Harris Corner Response (filter top N keypoints)
        ├── Step 3.1: Compute Gradients Ix, Iy
        ├── Step 3.2: Build Structure Tensor M
        └── Step 3.3: Compute Corner Response R
        ↓
Step 4: Orientation Assignment (intensity centroid)
        ├── Step 4.1: Compute Image Moments
        ├── Step 4.2: Compute Centroid
        └── Step 4.3: Compute Angle θ
        ↓
OUTPUT: Keypoints with (x, y, scale, orientation)
```

---
## Step 1: Scale-Space Pyramid

**Why?** Detect features at multiple scales for scale invariance.

### The Mathematics

**Scale at level n:**
$$scale_n = \frac{1}{f^n}$$

where:
- f = 1.2 (scale factor, default)
- n = pyramid level (0, 1, 2, ..., 7)

**Image size at level n:**
$$W_n = W_0 \times scale_n = W_0 / f^n$$
$$H_n = H_0 \times scale_n = H_0 / f^n$$

### Detailed Calculation Example

```
For original image 640 × 480:

Level   Scale Formula        Scale Value    Image Size
─────────────────────────────────────────────────────────
  0     1/1.2⁰ = 1/1        1.000          640 × 480
  1     1/1.2¹ = 1/1.2      0.833          533 × 400
  2     1/1.2² = 1/1.44     0.694          444 × 333
  3     1/1.2³ = 1/1.728    0.579          370 × 278
  4     1/1.2⁴ = 1/2.074    0.482          309 × 231
  5     1/1.2⁵ = 1/2.488    0.402          257 × 193
  6     1/1.2⁶ = 1/2.986    0.335          214 × 161
  7     1/1.2⁷ = 1/3.583    0.279          179 × 134
```

### Key Difference from SIFT

| Property | SIFT | ORB |
|----------|------|-----|
| Scale factor | 2.0 (per octave) | 1.2 (per level) |
| Method | Gaussian blur + downsample | Direct downsample |
| Levels | 4 octaves × 5 scales | 8 levels |
| Total images | 20 Gaussian + 16 DoG | 8 pyramid levels |

![Pyramid Structure](images/orb_pyramid_structure.png)

![Step 1: Full Pyramid](images/orb_step1_full_pyramid.png)

In [ ]:
def build_scale_pyramid(img, n_levels=8, scale_factor=1.2):
    """
    Build image pyramid for multi-scale detection.
    
    scale_n = 1 / f^n
    
    Parameters:
    - img: Input grayscale image (normalized 0-1)
    - n_levels: Number of pyramid levels (default: 8)
    - scale_factor: Scale ratio between levels (default: 1.2)
    
    Returns:
    - pyramid: List of images at different scales
    - scales: List of scale values
    """
    pyramid = [img.copy()]
    scales = [1.0]
    
    for level in range(1, n_levels):
        scale = 1.0 / (scale_factor ** level)
        new_h = int(img.shape[0] * scale)
        new_w = int(img.shape[1] * scale)
        
        if new_h < 10 or new_w < 10:
            break
        
        pil_img = Image.fromarray((img * 255).astype(np.uint8))
        pil_resized = pil_img.resize((new_w, new_h), Image.LANCZOS)
        resized = np.array(pil_resized) / 255.0
        
        pyramid.append(resized)
        scales.append(scale)
    
    return pyramid, scales

# Build pyramid
pyramid, scales = build_scale_pyramid(img)

print(f"Pyramid built with {len(pyramid)} levels")
for i, (p, s) in enumerate(zip(pyramid, scales)):
    print(f"  Level {i}: {p.shape[1]}×{p.shape[0]} (scale={s:.3f})")

---
## Step 2: FAST Corner Detection

**Why?** FAST (Features from Accelerated Segment Test) is extremely fast for real-time applications.

### The Bresenham Circle (16 pixels)

The FAST detector uses a circle of 16 pixels around each candidate point:

```
         16  1   2
      15          3
    14              4
    13       p      5
    12              6
      11          7
         10  9   8
```

![FAST Circle](images/orb_fast_circle.png)

### Circle Pixel Offsets

```
Position  Offset (dx, dy)    Position  Offset (dx, dy)
────────────────────────────────────────────────────────
   1      ( 0, -3)              9      ( 0, +3)
   2      (+1, -3)             10      (-1, +3)
   3      (+2, -2)             11      (-2, +2)
   4      (+3, -1)             12      (-3, +1)
   5      (+3,  0)             13      (-3,  0)
   6      (+3, +1)             14      (-3, -1)
   7      (+2, +2)             15      (-2, -2)
   8      (+1, +3)             16      (-1, -3)
```

### FAST-9 Algorithm

**Step 2.1: Define threshold**

```
Let I_p = intensity of center pixel p
Let t = threshold (default: 0.08 for normalized [0,1] images)

Upper bound: I_p + t
Lower bound: I_p - t
```

**Step 2.2: High-Speed Test (Cardinal Points)**

Before checking all 16 pixels, test positions 1, 5, 9, 13 (N, E, S, W):

```
         1 (North)
         ↓
    13 ← p → 5
  (West)     (East)
         ↓
         9 (South)

Rule:
  If at least 3 of {1, 5, 9, 13} are 'B' (Brighter) → Continue
  If at least 3 of {1, 5, 9, 13} are 'D' (Darker)   → Continue
  Otherwise → REJECT (cannot have 9 contiguous)
```

**Why this works:** For 9 contiguous pixels to be brighter (or darker), at least 3 of the 4 cardinal points MUST be included.

**Step 2.3: Classify Each Circle Pixel**

```
For each of the 16 circle pixels with intensity I_c:

  If I_c > I_p + t  →  Label = 'B' (Brighter)
  If I_c < I_p - t  →  Label = 'D' (Darker)
  Otherwise         →  Label = 'S' (Similar)
```

**Step 2.4: Check Contiguous Condition**

```
CORNER if: 9+ contiguous pixels are ALL 'B'
       OR: 9+ contiguous pixels are ALL 'D'

Note: "Contiguous" wraps around (pixel 16 is adjacent to pixel 1)
```

![Step 2: FAST Detail](images/orb_step2_detail.png)

![All FAST Corners](images/orb_step2_all_fast.png)

In [ ]:
# Bresenham circle offsets (16 pixels around center)
CIRCLE_OFFSETS = [
    (0, -3), (1, -3), (2, -2), (3, -1),    # positions 1-4
    (3, 0), (3, 1), (2, 2), (1, 3),         # positions 5-8
    (0, 3), (-1, 3), (-2, 2), (-3, 1),      # positions 9-12
    (-3, 0), (-3, -1), (-2, -2), (-1, -3)   # positions 13-16
]

def fast_corner_test(img, x, y, threshold=0.08, n_contiguous=9):
    """
    FAST corner test at a single pixel.
    
    CORNER if: 9+ contiguous pixels are ALL brighter or ALL darker
    """
    h, w = img.shape
    
    if x < 3 or x >= w - 3 or y < 3 or y >= h - 3:
        return False, 0
    
    center = img[y, x]
    upper = center + threshold
    lower = center - threshold
    
    # High-speed test: check positions 1, 5, 9, 13 first
    test_positions = [0, 4, 8, 12]
    n_brighter = sum(1 for pos in test_positions if img[y + CIRCLE_OFFSETS[pos][1], x + CIRCLE_OFFSETS[pos][0]] > upper)
    n_darker = sum(1 for pos in test_positions if img[y + CIRCLE_OFFSETS[pos][1], x + CIRCLE_OFFSETS[pos][0]] < lower)
    
    if n_brighter < 3 and n_darker < 3:
        return False, 0
    
    # Full 16-pixel test
    labels = []
    for dx, dy in CIRCLE_OFFSETS:
        val = img[y + dy, x + dx]
        if val > upper:
            labels.append('B')
        elif val < lower:
            labels.append('D')
        else:
            labels.append('S')
    
    # Check for contiguous pixels (with wrap-around)
    labels_extended = labels + labels
    max_b = max_d = 0
    cnt_b = cnt_d = 0
    
    for label in labels_extended:
        if label == 'B':
            cnt_b += 1
            cnt_d = 0
            max_b = max(max_b, cnt_b)
        elif label == 'D':
            cnt_d += 1
            cnt_b = 0
            max_d = max(max_d, cnt_d)
        else:
            cnt_b = cnt_d = 0
    
    max_b = min(max_b, 16)
    max_d = min(max_d, 16)
    response = max(max_b, max_d)
    
    return (max_b >= n_contiguous or max_d >= n_contiguous), response

def detect_fast_corners(img, threshold=0.08, n_contiguous=9):
    """Detect FAST corners in image."""
    h, w = img.shape
    keypoints = []
    
    for y in range(3, h - 3):
        for x in range(3, w - 3):
            is_corner, response = fast_corner_test(img, x, y, threshold, n_contiguous)
            if is_corner:
                keypoints.append({'x': x, 'y': y, 'response': response})
    
    return keypoints

# Detect FAST corners
fast_keypoints = detect_fast_corners(img)
print(f"Detected {len(fast_keypoints)} FAST corners")

---
## Step 3: Harris Corner Response

**Why?** FAST finds many corners, but Harris filter keeps only the strongest and most stable ones.

### Step 3.1: Compute Image Gradients

Using Sobel operators:

```
       ┌─────┬─────┬─────┐         ┌─────┬─────┬─────┐
       │ -1  │  0  │ +1  │         │ -1  │ -2  │ -1  │
       ├─────┼─────┼─────┤         ├─────┼─────┼─────┤
Sx =   │ -2  │  0  │ +2  │   Sy =  │  0  │  0  │  0  │
       ├─────┼─────┼─────┤         ├─────┼─────┼─────┤
       │ -1  │  0  │ +1  │         │ +1  │ +2  │ +1  │
       └─────┴─────┴─────┘         └─────┴─────┴─────┘

Ix = Sx * I    (horizontal gradient)
Iy = Sy * I    (vertical gradient)
```

### Step 3.2: Build Structure Tensor

$$I_{xx} = Gaussian\_blur(I_x \times I_x)$$
$$I_{yy} = Gaussian\_blur(I_y \times I_y)$$
$$I_{xy} = Gaussian\_blur(I_x \times I_y)$$

**Structure Tensor:**
$$M = \begin{bmatrix} I_{xx} & I_{xy} \\ I_{xy} & I_{yy} \end{bmatrix}$$

### Step 3.3: Compute Corner Response

$$R = det(M) - k \times trace(M)^2$$

where:
- $det(M) = I_{xx} \times I_{yy} - I_{xy}^2$
- $trace(M) = I_{xx} + I_{yy}$
- k = 0.04 (Harris constant)

### Interpretation of Harris Response

| Response R | Meaning | Action |
|------------|---------|--------|
| R >> 0 | Strong corner | ✓ KEEP |
| R ≈ 0 | Flat region | ✗ REJECT |
| R << 0 | Edge | ✗ REJECT |

![Harris Substeps](images/orb_step3_substeps.png)

![All Harris Corners](images/orb_step3_all_harris.png)

In [ ]:
def compute_harris_response(img, keypoints, k=0.04, block_size=3):
    """
    Compute Harris corner response for each FAST keypoint.
    
    R = det(M) - k × trace(M)²
    """
    Ix = sobel(img, axis=1)
    Iy = sobel(img, axis=0)
    
    Ixx = Ix * Ix
    Iyy = Iy * Iy
    Ixy = Ix * Iy
    
    sigma = block_size / 3.0
    Ixx = gaussian_filter(Ixx, sigma)
    Iyy = gaussian_filter(Iyy, sigma)
    Ixy = gaussian_filter(Ixy, sigma)
    
    h, w = img.shape
    
    for kp in keypoints:
        x, y = kp['x'], kp['y']
        if 0 <= x < w and 0 <= y < h:
            det_M = Ixx[y, x] * Iyy[y, x] - Ixy[y, x] ** 2
            trace_M = Ixx[y, x] + Iyy[y, x]
            kp['harris'] = det_M - k * (trace_M ** 2)
        else:
            kp['harris'] = 0
    
    return keypoints

def non_maximum_suppression(keypoints, radius=3):
    """Non-maximum suppression to keep only local maxima."""
    if len(keypoints) == 0:
        return []
    
    keypoints = sorted(keypoints, key=lambda k: k.get('harris', 0), reverse=True)
    kept = []
    suppressed = set()
    
    for i, kp in enumerate(keypoints):
        if i in suppressed:
            continue
        kept.append(kp)
        for j, other in enumerate(keypoints):
            if j != i and j not in suppressed:
                dist = np.sqrt((kp['x'] - other['x'])**2 + (kp['y'] - other['y'])**2)
                if dist < radius:
                    suppressed.add(j)
    
    return kept

# Compute Harris response
harris_keypoints = compute_harris_response(img, fast_keypoints)
harris_keypoints = non_maximum_suppression(harris_keypoints)
harris_keypoints = sorted(harris_keypoints, key=lambda k: k.get('harris', 0), reverse=True)[:500]
print(f"After Harris filtering: {len(harris_keypoints)} keypoints")

---
## Step 4: Orientation Assignment (Intensity Centroid)

**Why?** Assign a consistent orientation to each keypoint for rotation invariance.

### Step 4.1: Compute Image Moments

For a circular patch of radius r around keypoint:

$$m_{10} = \sum x \times I(x, y)$$
$$m_{01} = \sum y \times I(x, y)$$
$$m_{00} = \sum I(x, y)$$

where (x, y) are relative to keypoint center

### Step 4.2: Compute Centroid

$$Centroid\ C = \left(\frac{m_{10}}{m_{00}}, \frac{m_{01}}{m_{00}}\right)$$

This is the "center of mass" of the intensity distribution

### Step 4.3: Compute Orientation Angle

$$\theta = atan2(m_{01}, m_{10})$$

This angle points from keypoint center toward the intensity centroid

### Why Intensity Centroid Works

```
Key insight: The centroid direction is STABLE under rotation

Original image:         Rotated 30°:

    ┌─────────┐            ╲────────╲
    │ Light   │             ╲Light  ╲
    │    C    │              ╲  C   ╲
    │   ↗     │               ╲ ↗   ╲
    │  ○      │                ○     ╲
    │ Dark    │                ╲Dark  ╲
    └─────────┘                 ╲──────╲

    θ = 45°                    θ = 75° (= 45° + 30°)

The centroid direction rotates WITH the image!
```

![Orientation Substeps](images/orb_step4_substeps.png)

![Orientation with Arrows](images/orb_step4_with_orientation.png)

In [ ]:
def compute_orientation(img, keypoints, patch_radius=15):
    """
    Compute orientation using intensity centroid method.
    
    θ = atan2(m₀₁, m₁₀)
    """
    h, w = img.shape
    
    for kp in keypoints:
        cx, cy = int(kp['x']), int(kp['y'])
        m10 = 0
        m01 = 0
        
        for dy in range(-patch_radius, patch_radius + 1):
            for dx in range(-patch_radius, patch_radius + 1):
                x, y = cx + dx, cy + dy
                if 0 <= x < w and 0 <= y < h:
                    intensity = img[y, x]
                    m10 += dx * intensity
                    m01 += dy * intensity
        
        theta = np.arctan2(m01, m10)
        kp['orientation'] = theta
    
    return keypoints

# Compute orientations
oriented_keypoints = compute_orientation(img, harris_keypoints)
print(f"Computed orientation for {len(oriented_keypoints)} keypoints")

---
# ✅ Detection Phase Complete!

We now have keypoints with (x, y, orientation).

---
# 2️⃣ DESCRIPTION PHASE

**Goal**: Create a compact binary descriptor that is invariant to rotation.

```
INPUT: 500 keypoints with (x, y, scale, orientation θ)
        ↓
Step 5: Compute rBRIEF Descriptors
        ├── Step 5.1: Generate sampling pattern (256 pairs)
        ├── Step 5.2: Rotate pattern by θ
        └── Step 5.3: Binary intensity comparisons
        ↓
OUTPUT: 500 × 256-bit descriptors
```

---
## Step 5: rBRIEF Descriptor

**Why rBRIEF?** BRIEF is fast (binary), but not rotation-invariant. rBRIEF (Rotated BRIEF) rotates the sampling pattern by the keypoint orientation θ.

### Step 5.1: Generate Sampling Pattern

256 pairs of points (p, q) within a 31×31 patch:

```
For each of 256 pairs:
  p_x, p_y ~ N(0, (31/2)²/25)  → Gaussian distribution
  q_x, q_y ~ N(0, (31/2)²/25)
  Clamp to [-15, 15]
```

![BRIEF Concept](images/orb_brief_concept.png)

### Step 5.2: Rotate Pattern by Orientation θ

**Rotation matrix:**
$$R(\theta) = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}$$

For each point pair (p, q):
$$p' = R(\theta) \times p$$
$$q' = R(\theta) \times q$$

![rBRIEF Rotation](images/orb_rbrief_rotation.png)

### Step 5.3: Binary Intensity Comparisons

```
For each of 256 rotated pairs:

  bit_i = 1  if I(p'_i) < I(q'_i)
          0  otherwise
```

### Why Binary Descriptors?

| Property | SIFT | ORB |
|----------|------|-----|
| Size | 512 bytes (128 × 4) | 32 bytes (256 / 8) |
| Matching | Euclidean dist | Hamming distance |
| Speed | ~100 μs/match | ~0.5 μs/match |

![Descriptor Substeps](images/orb_step5_substeps.png)

![Keypoints with Descriptors](images/orb_step5_with_descriptors.png)

In [ ]:
def generate_brief_pattern(n_pairs=256, patch_size=31, seed=42):
    """Generate BRIEF sampling pattern."""
    np.random.seed(seed)
    half = patch_size // 2
    pattern = []
    
    for _ in range(n_pairs):
        p_x = int(np.clip(np.random.randn() * half / 2, -half, half))
        p_y = int(np.clip(np.random.randn() * half / 2, -half, half))
        q_x = int(np.clip(np.random.randn() * half / 2, -half, half))
        q_y = int(np.clip(np.random.randn() * half / 2, -half, half))
        pattern.append(((p_x, p_y), (q_x, q_y)))
    
    return pattern

def rotate_pattern(pattern, theta):
    """Rotate BRIEF pattern by angle theta."""
    cos_t = np.cos(theta)
    sin_t = np.sin(theta)
    
    rotated = []
    for (px, py), (qx, qy) in pattern:
        rpx = int(round(px * cos_t - py * sin_t))
        rpy = int(round(px * sin_t + py * cos_t))
        rqx = int(round(qx * cos_t - qy * sin_t))
        rqy = int(round(qx * sin_t + qy * cos_t))
        rotated.append(((rpx, rpy), (rqx, rqy)))
    
    return rotated

def compute_rbrief_descriptor(img, keypoint, pattern):
    """Compute rBRIEF descriptor for a keypoint."""
    h, w = img.shape
    cx, cy = int(keypoint['x']), int(keypoint['y'])
    theta = keypoint.get('orientation', 0)
    
    rotated_pattern = rotate_pattern(pattern, theta)
    descriptor = np.zeros(len(pattern), dtype=np.uint8)
    
    for i, ((px, py), (qx, qy)) in enumerate(rotated_pattern):
        p_x, p_y = cx + px, cy + py
        q_x, q_y = cx + qx, cy + qy
        
        if 0 <= p_x < w and 0 <= p_y < h and 0 <= q_x < w and 0 <= q_y < h:
            if img[p_y, p_x] < img[q_y, q_x]:
                descriptor[i] = 1
    
    return descriptor

def extract_descriptors(img, keypoints, n_pairs=256):
    """Extract rBRIEF descriptors for all keypoints."""
    pattern = generate_brief_pattern(n_pairs)
    descriptors = []
    valid_keypoints = []
    
    h, w = img.shape
    
    for kp in keypoints:
        x, y = int(kp['x']), int(kp['y'])
        
        if x < 16 or x >= w - 16 or y < 16 or y >= h - 16:
            continue
        
        desc = compute_rbrief_descriptor(img, kp, pattern)
        descriptors.append(desc)
        kp['descriptor'] = desc
        valid_keypoints.append(kp)
    
    return valid_keypoints, np.array(descriptors)

# Extract descriptors
final_keypoints, descriptors = extract_descriptors(img, oriented_keypoints)
print(f"Extracted {len(descriptors)} rBRIEF descriptors (256-bit each)")

---
## Step 6: Hamming Distance Matching

**Why Hamming?** For binary descriptors, Hamming distance (number of differing bits) can be computed extremely fast using XOR and population count.

### The Mathematics

$$Hamming(A, B) = popcount(A\ XOR\ B)$$

where:
- XOR: bitwise exclusive-or
- popcount: count of 1-bits

### Matching Threshold

```
For 256-bit descriptors:
  Maximum distance = 256 (all bits different)
  Minimum distance = 0 (identical)

Typical threshold:
  distance < 50 → GOOD MATCH
  distance ≥ 50 → NOT A MATCH

Interpretation: ~19.5% of bits can differ
```

![Hamming Distance](images/orb_hamming_distance.png)

![Matching Visualization](images/orb_step6_matching.png)

In [ ]:
def hamming_distance(desc1, desc2):
    """Compute Hamming distance between two binary descriptors."""
    return np.sum(desc1 != desc2)

# Demo: compute distances between first few descriptors
if len(descriptors) >= 5:
    print("Hamming distances between first 5 descriptors:")
    print("     ", end="")
    for j in range(5):
        print(f"  D{j}  ", end="")
    print()
    
    for i in range(5):
        print(f"D{i}  ", end="")
        for j in range(5):
            dist = hamming_distance(descriptors[i], descriptors[j])
            print(f" {dist:3d}  ", end="")
        print()

---
# Complete Pipeline Summary

```
┌─────────────────────────────────────────────────────────────────────┐
│                    ORB Feature Extraction Pipeline                   │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  INPUT: Image (640 × 480)                                           │
│           │                                                         │
│           ▼                                                         │
│  STEP 1: Scale-Space Pyramid (8 levels, factor 1.2)                 │
│           │                                                         │
│           ▼                                                         │
│  STEP 2: FAST Corner Detection → 10,510 corners                     │
│           │                                                         │
│           ▼                                                         │
│  STEP 3: Harris Corner Response → Top 500 keypoints                 │
│           │                                                         │
│           ▼                                                         │
│  STEP 4: Orientation Assignment → 500 oriented keypoints            │
│           │                                                         │
│           ▼                                                         │
│  STEP 5: rBRIEF Descriptor → 500 × 256-bit descriptors              │
│           │                                                         │
│           ▼                                                         │
│  STEP 6: Hamming Matching → Matched keypoint pairs                  │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

![Final Result](images/orb_final_result.png)

![Complete Pipeline](images/orb_complete_pipeline.png)

---
## ORB vs SIFT Comparison

| Feature | SIFT | ORB |
|---------|------|-----|
| **Speed** | ~100 ms | ~10 ms |
| **Descriptor size** | 512 bytes (128 floats) | 32 bytes (256 bits) |
| **Matching speed** | Slow (Euclidean) | Fast (Hamming) |
| **Scale invariance** | ✓ (DoG pyramid) | ✓ (direct pyramid) |
| **Rotation invariance** | ✓ (gradient histogram) | ✓ (intensity centroid) |
| **Patent** | Was patented | Patent-free |
| **Best for** | Accuracy-critical | Real-time applications |

![ORB vs SIFT](images/orb_sift_comparison.png)

## Quick Reference: All Formulas

### Detection Phase

| Formula | Description |
|---------|-------------|
| $scale_n = 1 / 1.2^n$ | Scale Pyramid |
| $I_c > I_p + t$ | FAST Brighter test |
| $I_c < I_p - t$ | FAST Darker test |
| $R = det(M) - k \times trace(M)^2$ | Harris Response |
| $\theta = atan2(m_{01}, m_{10})$ | Orientation |

### Description Phase

| Formula | Description |
|---------|-------------|
| $p' = R(\theta) \times p$ | Pattern Rotation |
| $bit_i = 1$ if $I(p'_i) < I(q'_i)$ | Binary Test |
| $d = popcount(A\ XOR\ B)$ | Hamming Distance |

In [ ]:
# Print final statistics
print("=" * 60)
print("ORB ALGORITHM COMPLETE")
print("=" * 60)
print(f"\nInput Image: {W} × {H} pixels")
print(f"\n1️⃣ DETECTION PHASE:")
print(f"   Step 1: Scale Pyramid - {len(pyramid)} levels")
print(f"   Step 2: FAST Corners - {len(fast_keypoints)} detected")
print(f"   Step 3: Harris Filter - {len(harris_keypoints)} keypoints")
print(f"   Step 4: Orientation - {len(oriented_keypoints)} with θ")
print(f"\n2️⃣ DESCRIPTION PHASE:")
print(f"   Step 5: rBRIEF - {len(descriptors)} × 256-bit descriptors")
print(f"\nFINAL OUTPUT: {len(final_keypoints)} keypoints with 256-bit descriptors")
print(f"Total storage: {len(final_keypoints) * 32} bytes")
print("=" * 60)

---
## References

1. Rublee, E., et al. (2011). "ORB: An efficient alternative to SIFT or SURF." ICCV.
2. Rosten, E., & Drummond, T. (2006). "Machine learning for high-speed corner detection." ECCV.
3. Calonder, M., et al. (2010). "BRIEF: Binary Robust Independent Elementary Features." ECCV.
4. Harris, C., & Stephens, M. (1988). "A combined corner and edge detector." Alvey Vision Conference.